# 1. Build the datasets (CPU, no GPU needed)

Runs the kit's generators in the cloud and writes the files you upload to Adaption.

**Settings:** Accelerator = None, Internet = On. Edit `REPO` below, then *Run All* (about 2 minutes).

Your own data goes in the GitHub repo first (edit in the GitHub web editor): `data/seeds/manual_listings.csv` (real messages), `data/seeds/advice_mcq.csv` and `data/seeds/hygiene_mcq.csv` (MCQs), `data/seeds/symptom_table.csv`.

## Data sources

| File | What it is | Used for |
| --- | --- | --- |
| `data/lexicon/agro_lexicon.json` | 42 products with French, English, Pidgin and local names; units with kg weights; towns in all 10 regions; quality words | every synthetic agro message. Price ranges there are illustrative, not market facts |
| `data/seeds/wfp_food_prices_cmr.csv` | WFP / HDX *Cameroon - Food Prices* (real market prices, XAF) | the reference prices in `agro_unit_price` |
| `data/seeds/wfp_markets_cmr.csv` | WFP / HDX *Cameroon - Markets* (market names, regions, coordinates) | reference only |
| `data/seeds/manual_listings.csv` | your real WhatsApp/Facebook messages, anonymised (target 20 per language) | half go to the `test_real` split |
| `data/seeds/advice_mcq.csv`, `data/seeds/hygiene_mcq.csv` | your MCQs from the fiches (target 40-80) | test set only, never training |
| `data/seeds/symptom_table.csv` | crop symptoms and actions | agro knowledge examples |
| `data/lexicon/pcm_review_sheet.csv` | every Pidgin string, for one native speaker to check | not used by the build; copy corrections back into the lexicon |

If the two WFP files are not in your repo, the next cell downloads them from HDX. Commit them to GitHub so every run uses the same prices: new HDX data changes the reference prices in the examples.


In [ ]:
REPO = 'https://github.com/YOUR_GITHUB_USER/nema-agro-adaptive.git'  # your public kit repo
!git clone -q $REPO kit
%cd kit

## Check the data sources

In [ ]:
# Gets the kit code. Works with Internet OFF if the GitHub-linked Kaggle dataset is attached (Add Input).
import glob, os, shutil, subprocess
from pathlib import Path

REPO = 'https://github.com/ngounou-tp/nema-agro-adaptive.git'  # used only when Internet is on
KIT = Path('/kaggle/working/kit')
if KIT.exists():
    shutil.rmtree(KIT)

found = sorted(glob.glob('/kaggle/input/**/scripts/build_all.sh', recursive=True))
if found:
    src = Path(found[0]).parents[1]
    shutil.copytree(src, KIT)
    subprocess.run(['chmod', '-R', 'u+w', str(KIT)], check=True)  # /kaggle/input is read-only
    print('kit copied from the attached Kaggle dataset:', src)
else:
    r = subprocess.run(['git', 'clone', '-q', '--depth', '1', REPO, str(KIT)], capture_output=True, text=True)
    if r.returncode:
        raise RuntimeError('Could not get the kit: no attached dataset contains scripts/build_all.sh and git clone failed '
                           '(Internet is off). Click Add Input and attach the dataset made from the GitHub repo.\n' + r.stderr)
    print('kit cloned from', REPO)

os.chdir(KIT)
print('working in', os.getcwd())

## Build

In [ ]:
!bash scripts/build_all.sh

In [ ]:
with open('data/build/gen_agro_calc.jsonl', encoding='utf-8') as f:
    n_wfp = sum('source WFP/HDX' in line for line in f)
print('agro_unit_price examples with real WFP reference prices:', n_wfp)
assert n_wfp, 'no WFP reference prices were used: check the WFP cell above'


In [ ]:
import json, shutil, os
os.makedirs('/kaggle/working/for_adaption', exist_ok=True)
for p in ('agro', 'waste'):
    shutil.copy(f'data/build/{p}/adaption_seed.csv', f'/kaggle/working/for_adaption/{p}_adaption_seed.csv')
    shutil.copy(f'data/build/{p}/test.jsonl', f'/kaggle/working/for_adaption/{p}_test_DO_NOT_UPLOAD.jsonl')
    s = json.load(open(f'data/build/{p}/stats.json'))
    print(p, 'train', s['train'], 'test', s['test'], 'leaks', s['leaks'])
print(os.listdir('/kaggle/working/for_adaption'))

**Next:** open the **Output** panel (right side), download `agro_adaption_seed.csv` and `waste_adaption_seed.csv`, and upload them in the Adaption app (playbook section 7). Never upload the `*_test_DO_NOT_UPLOAD.jsonl` files.

If you changed seeds, commit the rebuilt `data/build/agro/test.jsonl` to GitHub too (download it from `kit/data/build/agro/`).